In [13]:
# Verify the current python environment
import sys
print(sys.executable)

/Volumes/Extreme Pro/FKG/SHACL-API-Docker/spacy_env/bin/python


In [14]:
import spacy
import pandas as pd
from rdflib import Graph, Literal
from rdflib.plugins.sparql import prepareQuery

# --- CONFIGURATION PATHS ---
LEXICAL_GRAPH_PATH = "/Volumes/Extreme Pro/FKG/SHACL-API-Docker/lexical.ttl"
STRUCTURAL_GRAPH_PATH = "/Volumes/Extreme Pro/FKG/SHACL-API-Docker/structural.ttl"
CSV_PATH = "Situation Shapes Workbook - Shape stubs.csv"

print("Loading local RDF graphs into memory...")
g = Graph()
g.parse(LEXICAL_GRAPH_PATH, format="turtle")
g.parse(STRUCTURAL_GRAPH_PATH, format="turtle")
print(f"Loaded {len(g)} triples.")

print("Loading spaCy NLP model...")
nlp = spacy.load("en_core_web_sm")

print("Loading Shape Translation Matrix...")
df = pd.read_csv(CSV_PATH)
df['shape_clean'] = df['3'].astype(str).str.replace(' ', '_')
print(f"Loaded {len(df)} shapes from matrix.")

Loading local RDF graphs into memory...
Loaded 61591 triples.
Loading spaCy NLP model...
Loading Shape Translation Matrix...
Loaded 91 shapes from matrix.


In [15]:
# --- PHASE 2: EVOCATION QUERY ---
evocation_query = prepareQuery("""
    PREFIX : <https://falcontologist.github.io/shacl-demo/ontology/>
    PREFIX sh: <http://www.w3.org/ns/shacl#>

    SELECT DISTINCT ?shape ?roleName
    WHERE {
        ?lemmaNode :lemma ?target_lemma .
        
        {
            ?lemmaNode :evokes ?shape .
        } UNION {
            ?lemmaNode :sense ?synset .
            ?synset :evokes ?shape .
        }
        
        ?shape sh:property ?propNode .
        ?propNode sh:name ?roleName .
    }
""")

# --- PHASE 3: MAPPING HEURISTIC ---
PROPBANK_TO_VERBNET = {
    "ARG0": ["Agent", "Pivot", "Causer", "Experiencer", "Stimulus"],
    "ARG1": ["Theme", "Patient", "Topic", "Attribute", "Material", "Asset"],
    "ARG2": ["Recipient", "Destination", "Instrument", "Beneficiary", "Goal", "Result"],
    "ARG3": ["Asset", "Source", "Initial_location", "Initial_state"],
    "ARG4": ["Destination", "Final_State"],
    "ARGM-LOC": ["Location"],
    "ARGM-DIR": ["Trajectory", "Source", "Destination"],
    "ARGM-MNR": ["Manner"],
    "ARGM-PRP": ["Eventuality", "Result"] 
}

def extract_semantic_roles(shape_name, srl_output, translation_matrix):
    shape_row = translation_matrix[translation_matrix['shape_clean'] == shape_name]
    if shape_row.empty:
        return {"Error": f"Shape '{shape_name}' not found in CSV matrix."}
        
    shape_dict = shape_row.dropna(axis=1).iloc[0].to_dict()
    extraction = {}
    
    for arg, text in srl_output.items():
        if arg == "V":
            continue
            
        candidate_vn_roles = PROPBANK_TO_VERBNET.get(arg, [])
        mapped = False
        
        for vn_role in candidate_vn_roles:
            if vn_role in shape_dict:
                shacl_role = shape_dict[vn_role]
                extraction[shacl_role] = text
                mapped = True
                break
                
        if not mapped:
            extraction[f"UNMAPPED_{arg}"] = text
            
    return extraction

In [16]:
# --- CELL 3: END-TO-END PIPELINE (INTERACTIVE) ---

def mock_srl_inference(sentence):
    mock_db = {
        "Microsoft acquired Nuance Communications for $19 billion.": {"ARG0": "Microsoft", "V": "acquired", "ARG1": "Nuance Communications", "ARG3": "for $19 billion"},
        "Fort Knox holds the United States' official gold reserves.": {"ARG0": "Fort Knox", "V": "holds", "ARG1": "the United States' official gold reserves"},
        "Walmart throws away approximately 1.5 billion pounds of unsold fresh produce every year.": {"ARG0": "Walmart", "V": "throws away", "ARG1": "approximately 1.5 billion pounds of unsold fresh produce", "ARGM-TMP": "every year"}
    }
    return mock_db.get(sentence, {})

test_sentences = [
    "Microsoft acquired Nuance Communications for $19 billion.",
    "Fort Knox holds the United States' official gold reserves.",
    "Walmart throws away approximately 1.5 billion pounds of unsold fresh produce every year."
]

print("--- RUNNING END-TO-END PIPELINE (PHASE 2 & 3) ---\n")

for sentence in test_sentences:
    print(f"Text: \"{sentence}\"")
    
    # 1. EVOCATION (Find Verbs)
    doc = nlp(sentence)
    target_lemmas = []
    for token in doc:
        if token.pos_ == "VERB" or token.dep_ == "ROOT":
            if token.lemma_.lower() not in target_lemmas:
                target_lemmas.append(token.lemma_.lower())
                
    if not target_lemmas:
        print("  -> No verbs found. Skipping.\n")
        continue

    # 2. RESOLVE SHAPES
    for target_lemma in target_lemmas:
        results = g.query(evocation_query, initBindings={'target_lemma': Literal(target_lemma)})
        evocations = {}
        
        for row in results:
            shape_clean = str(row.shape).split('/')[-1].split('#')[-1].replace('_shape', '').replace('>', '').replace('<', '')
            role_name = "PPT" if str(row.roleName).lower() == "proto-patient" else str(row.roleName)
            if shape_clean not in evocations: evocations[shape_clean] = []
            evocations[shape_clean].append(role_name)
            
        if not evocations: continue
        
        # INTERACTIVE DISAMBIGUATION
        selected_shape = list(evocations.keys())[0]
        if len(evocations) > 1:
            shapes_list = list(evocations.keys())
            print(f"  -> Multiple shapes evoked for '{target_lemma}'. Please select:")
            for idx, shape in enumerate(shapes_list):
                print(f"     {idx + 1}. {shape}")
                
            while True:
                try:
                    # NOTE: Look for the input box in your Jupyter UI!
                    choice = int(input(f"     Select 1-{len(shapes_list)}: "))
                    if 1 <= choice <= len(shapes_list):
                        selected_shape = shapes_list[choice - 1]
                        break
                    else:
                        print("     Invalid choice.")
                except ValueError:
                    print("     Invalid input. Enter a number.")
            
        print(f"  -> User-selected Shape: [{selected_shape}]")
        
        # 3. EXTRACTION (SRL -> SHACL)
        srl_raw = mock_srl_inference(sentence)
        if not srl_raw:
            print("  -> No SRL mock data available for this sentence.")
            continue
            
        final_roles = extract_semantic_roles(selected_shape, srl_raw, df)
        
        print("  -> Final Extracted Roles:")
        for role, text_chunk in final_roles.items():
            print(f"     * {role}: '{text_chunk}'")
    print("-" * 50)

--- RUNNING END-TO-END PIPELINE (PHASE 2 & 3) ---

Text: "Microsoft acquired Nuance Communications for $19 billion."
  -> User-selected Shape: [Dynamic_Possession]
  -> Final Extracted Roles:
     * transactor: 'Microsoft'
     * item: 'Nuance Communications'
     * asset: 'for $19 billion'
--------------------------------------------------
Text: "Fort Knox holds the United States' official gold reserves."
  -> Multiple shapes evoked for 'hold'. Please select:
     1. Capacity
     2. Constrain
     3. Judge
     4. Statement
     5. Causative_Creation
  -> User-selected Shape: [Constrain]
  -> Final Extracted Roles:
     * antagonist: 'Fort Knox'
     * agonist: 'the United States' official gold reserves'
--------------------------------------------------
Text: "Walmart throws away approximately 1.5 billion pounds of unsold fresh produce every year."
  -> Multiple shapes evoked for 'throw'. Please select:
     1. Affect
     2. Throw
     3. Carry
  -> User-selected Shape: [Throw]
  -

In [20]:
# --- CELL 4: TRAINING MODE & INITIAL LEARNING LOOP ---

import numpy as np
import pandas as pd

DIMENSIONS = [
    "physical", "bounded", "locative", "animate", "sentient", "volitional",
    "institutional", "collective", "telic", "symbolic", "scalar", "temporal"
]

# 1. THE UNTRAINED BASELINE (Blank Slates)
role_profiles = {
    "transactor": {"trained": False, "vector": [0.0] * 12},
    "antagonist": {"trained": False, "vector": [0.0] * 12},
    "thrower": {"trained": False, "vector": [0.0] * 12}
}

# Simulating a realistic, messy Virtuoso retrieval with diverse entity types
mock_candidates = {
    "Microsoft": {
        ":MICROSOFT_CORP.organization_entity": [0.1, 0.8, 0.1, 0.0, 0.0, 0.8, 0.9, 0.8, 0.4, 0.3, 0.0, 0.1],
        ":MSFT.equity_entity": [0.0, 0.9, 0.0, 0.0, 0.0, 0.0, 0.2, 0.0, 0.1, 0.9, 0.7, 0.2],
        ":MSFT_3_125_2025.corporate_bond_entity": [0.0, 0.9, 0.0, 0.0, 0.0, 0.0, 0.3, 0.0, 0.2, 0.9, 0.8, 0.9],
        ":WINDOWS_11.product_entity": [0.0, 0.8, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.9, 0.8, 0.0, 0.0]
    },
    "Fort Knox": {
        ":FORT_KNOX.organization_entity": [0.3, 0.8, 0.6, 0.0, 0.0, 0.7, 0.9, 0.7, 0.5, 0.4, 0.0, 0.1],
        ":FORT_KNOX_BASE.location_entity": [0.9, 0.9, 0.9, 0.0, 0.0, 0.0, 0.3, 0.0, 0.2, 0.1, 0.0, 0.1],
        ":FORT_KNOX_BULLION_DEPOSITORY.building_entity": [0.9, 0.9, 0.9, 0.0, 0.0, 0.0, 0.2, 0.0, 0.8, 0.1, 0.0, 0.0]
    },
    "Walmart": {
        ":WALMART_INC.organization_entity": [0.1, 0.8, 0.2, 0.0, 0.0, 0.8, 0.9, 0.9, 0.4, 0.3, 0.0, 0.1],
        ":WMT.equity_entity": [0.0, 0.9, 0.0, 0.0, 0.0, 0.0, 0.2, 0.0, 0.1, 0.9, 0.7, 0.2],
        ":WALMART_SUPERCENTER_STORE_314.location_entity": [0.9, 0.8, 0.9, 0.0, 0.0, 0.0, 0.4, 0.2, 0.6, 0.2, 0.0, 0.1],
        ":WALMART_RETAIL_INDEX.index_entity": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.9, 0.1, 0.9, 0.8, 0.1]
    }
}

# The parsed outputs from Phase 3, now carrying their sentence context
extracted_entities = [
    {
        "role": "transactor", 
        "string": "Microsoft", 
        "sentence": "Microsoft acquired Nuance Communications for $19 billion."
    },
    {
        "role": "antagonist", 
        "string": "Fort Knox", 
        "sentence": "Fort Knox holds the United States' official gold reserves."
    },
    {
        "role": "thrower", 
        "string": "Walmart", 
        "sentence": "Walmart throws away approximately 1.5 billion pounds of unsold fresh produce every year."
    }
]

print("--- INITIATING SYSTEM COLD-START TRAINING ---\n")

for extraction in extracted_entities:
    extracted_role = extraction["role"]
    extracted_string = extraction["string"]
    context_sentence = extraction["sentence"]
    
    print(f"[{extracted_role.upper()}] Triage Queue for '{extracted_string}'")
    print(f"Context: \"{context_sentence}\"")
    
    # 2. CHECK TRAINING STATUS
    is_trained = role_profiles[extracted_role]["trained"]
    if not is_trained:
        print(f"Status: UNTRAINED DEFAULT. Awaiting human-in-the-loop ground truth.")
    
    candidates = mock_candidates.get(extracted_string, {})
    candidate_uris = list(candidates.keys())
    
    # 3. INTERACTIVE HUMAN SELECTION
    print(f"Please select the correct URI context for '{extracted_string}':")
    for idx, uri in enumerate(candidate_uris):
        print(f"  {idx + 1}. {uri}")
        
    while True:
        try:
            choice = int(input(f"Select 1-{len(candidate_uris)}: "))
            if 1 <= choice <= len(candidate_uris):
                selected_uri = candidate_uris[choice - 1]
                break
            else:
                print("Invalid choice. Try again.")
        except ValueError:
            print("Invalid input. Enter a number.")
            
    print(f"\n[HUMAN VALIDATED TARGET]: {selected_uri}")
    validated_vector = np.array(candidates[selected_uri])
    
    # 4. APPLY THE LEARNING MATH
    old_vector = np.array(role_profiles[extracted_role]["vector"])
    
    if not is_trained:
        print(f"\n--- LEARNING MODE: CALIBRATING ROLE '{extracted_role}' ---")
        print("First training instance detected. Bootstrapping baseline centroid entirely from human selection...\n")
        new_vector = validated_vector
        role_profiles[extracted_role]["trained"] = True
        role_profiles[extracted_role]["vector"] = new_vector.tolist()
    
    # 5. GENERATE INTERPRETABILITY LEDGER
    learning_comparison = pd.DataFrame({
        "Dimension": DIMENSIONS,
        "Untrained_Default": np.round(old_vector, 4),
        "Human_Selected_Target": np.round(validated_vector, 4),
        "New_Learned_Centroid": np.round(new_vector, 4),
        "Delta": np.round(new_vector - old_vector, 4)
    })
    
    print(learning_comparison.to_string(index=False))
    print("-" * 75)
    print(f"The '{extracted_role}' role is now mathematically trained and primed.\n")

print("Cold-Start Training Complete.")

--- INITIATING SYSTEM COLD-START TRAINING ---

[TRANSACTOR] Triage Queue for 'Microsoft'
Context: "Microsoft acquired Nuance Communications for $19 billion."
Status: UNTRAINED DEFAULT. Awaiting human-in-the-loop ground truth.
Please select the correct URI context for 'Microsoft':
  1. :MICROSOFT_CORP.organization_entity
  2. :MSFT.equity_entity
  3. :MSFT_3_125_2025.corporate_bond_entity
  4. :WINDOWS_11.product_entity

[HUMAN VALIDATED TARGET]: :MICROSOFT_CORP.organization_entity

--- LEARNING MODE: CALIBRATING ROLE 'transactor' ---
First training instance detected. Bootstrapping baseline centroid entirely from human selection...

    Dimension  Untrained_Default  Human_Selected_Target  New_Learned_Centroid  Delta
     physical                0.0                    0.1                   0.1    0.1
      bounded                0.0                    0.8                   0.8    0.8
     locative                0.0                    0.1                   0.1    0.1
      animate       

In [21]:
# --- CELL 6: GRAPH WRITE-BACK (SPARQL UPDATE) ---

def generate_sparql_update(role_name, new_vector):
    """
    Generates a SPARQL DELETE/INSERT query to update the 12-dimensional 
    cognitive baseline for a specific role property in the graph.
    """
    ontology_prefix = "https://falcontologist.github.io/shacl-demo/ontology/"
    
    # 1. Build the predicate-value pairs for the INSERT clause
    insert_statements = []
    for dim, score in zip(DIMENSIONS, new_vector):
        insert_statements.append(f"        :{dim}Score \"{score:.4f}\"^^xsd:float ;")
    
    # Format the insert block, replacing the last semicolon with a period
    insert_block = "\n".join(insert_statements)[:-1] + "."
    
    # 2. Build the OPTIONAL blocks for the DELETE clause
    # We use OPTIONAL so the query doesn't fail if it's a blank slate (first time training)
    delete_statements = []
    where_statements = []
    for dim in DIMENSIONS:
        delete_statements.append(f"        :{dim}Score ?old_{dim} ;")
        where_statements.append(f"        OPTIONAL {{ :{role_name} :{dim}Score ?old_{dim} . }}")
        
    delete_block = "\n".join(delete_statements)[:-1] + "."
    where_block = "\n".join(where_statements)
    
    # 3. Assemble the full query
    query = f"""
    PREFIX : <{ontology_prefix}>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

    DELETE {{
        :{role_name} 
{delete_block}
    }}
    INSERT {{
        :{role_name} 
{insert_block}
    }}
    WHERE {{
{where_block}
    }}
    """
    return query

# Let's generate the query for our newly trained 'transactor'
new_transactor_vector = role_profiles["transactor"]["vector"]
sparql_query = generate_sparql_update("transactor", new_transactor_vector)

print("--- GENERATED SPARQL UPDATE QUERY ---\n")
print(sparql_query)

--- GENERATED SPARQL UPDATE QUERY ---


    PREFIX : <https://falcontologist.github.io/shacl-demo/ontology/>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

    DELETE {
        :transactor 
        :physicalScore ?old_physical ;
        :boundedScore ?old_bounded ;
        :locativeScore ?old_locative ;
        :animateScore ?old_animate ;
        :sentientScore ?old_sentient ;
        :volitionalScore ?old_volitional ;
        :institutionalScore ?old_institutional ;
        :collectiveScore ?old_collective ;
        :telicScore ?old_telic ;
        :symbolicScore ?old_symbolic ;
        :scalarScore ?old_scalar ;
        :temporalScore ?old_temporal .
    }
    INSERT {
        :transactor 
        :physicalScore "0.1000"^^xsd:float ;
        :boundedScore "0.8000"^^xsd:float ;
        :locativeScore "0.1000"^^xsd:float ;
        :animateScore "0.0000"^^xsd:float ;
        :sentientScore "0.0000"^^xsd:float ;
        :volitionalScore "0.8000"^^xsd:float ;
        :institutional

In [ ]:
# --- CELL 6: SEMI-AUTOMATED POST-TRAINING DISAMBIGUATION (WITH TRIAGE) ---

import numpy as np
import pandas as pd

# 1. SYSTEM SETTINGS
CONFIDENCE_THRESHOLD = 0.85

# Mocking the current state of our graph's semantic memory
role_profiles = {
    # Transactor and Antagonist were trained in Cell 4 & 5
    "transactor": {"trained": True, "vector": [0.1, 0.8, 0.1, 0.0, 0.0, 0.8, 0.9, 0.8, 0.4, 0.3, 0.0, 0.1]},
    "antagonist": {"trained": True, "vector": [0.9, 0.9, 0.9, 0.0, 0.0, 0.0, 0.2, 0.0, 0.8, 0.1, 0.0, 0.0]},
    "thrower": {"trained": True, "vector": [0.1, 0.8, 0.2, 0.0, 0.0, 0.8, 0.9, 0.9, 0.4, 0.3, 0.0, 0.1]},
    # Item is a brand new role we haven't trained yet
    "item": {"trained": False, "vector": [0.0] * 12} 
}

# 2. NEW EXTRACTIONS
post_training_extractions = [
    {
        "role": "transactor", 
        "string": "Amazon", 
        "sentence": "Amazon acquired Whole Foods for $13.7 billion."
    },
    {
        "role": "item", 
        "string": "Whole Foods", 
        "sentence": "Amazon acquired Whole Foods for $13.7 billion."
    },
    {
        "role": "thrower", 
        "string": "Target", 
        "sentence": "Target throws away millions of pounds of returned merchandise."
    }
]

# 3. MOCK GRAPH CANDIDATES
mock_new_candidates = {
    "Amazon": {
        ":AMAZON_COM.organization_entity": [0.1, 0.8, 0.1, 0.0, 0.0, 0.8, 0.9, 0.8, 0.4, 0.3, 0.0, 0.1],
        ":AMAZON_RIVER.location_entity": [0.9, 0.2, 0.9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.1, 0.0, 0.0]
    },
    "Whole Foods": {
        ":WHOLE_FOODS_MARKET.organization_entity": [0.1, 0.8, 0.2, 0.0, 0.0, 0.8, 0.9, 0.9, 0.4, 0.3, 0.0, 0.1],
        ":ORGANIC_FOOD.substance_entity": [0.9, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    },
    "Target": {
        # Notice we made Target Corp slightly fuzzy on volition/institutional to simulate ambiguity
        ":TARGET_CORP.organization_entity": [0.1, 0.8, 0.2, 0.0, 0.0, 0.4, 0.5, 0.8, 0.4, 0.3, 0.0, 0.1],
        ":TARGET_GOAL.conceptual_entity": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.9, 0.8, 0.0, 0.0]
    }
}

def cosine_similarity(v1, v2):
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    if norm_v1 == 0 or norm_v2 == 0: return 0.0
    return dot_product / (norm_v1 * norm_v2)

print(f"--- INITIATING SEMI-AUTOMATED PIPELINE (Threshold: {CONFIDENCE_THRESHOLD}) ---\n")

for extraction in post_training_extractions:
    extracted_role = extraction["role"]
    extracted_string = extraction["string"]
    context_sentence = extraction["sentence"]
    
    print(f"[{extracted_role.upper()}] String: '{extracted_string}'")
    print(f"Context: \"{context_sentence}\"")
    
    target_vector = np.array(role_profiles[extracted_role]["vector"])
    candidates = mock_new_candidates.get(extracted_string, {})
    candidate_uris = list(candidates.keys())
    
    requires_triage = False
    triage_reason = ""
    winner_uri = None
    winner_score = 0.0
    
    # 4. EVALUATE SYSTEM READINESS & CONFIDENCE
    if not role_profiles[extracted_role]["trained"]:
        requires_triage = True
        triage_reason = "Role is untrained (Cold-Start)."
    else:
        results = []
        for uri, vec_list in candidates.items():
            candidate_vector = np.array(vec_list)
            score = cosine_similarity(target_vector, candidate_vector)
            results.append((uri, score, candidate_vector))
            
        results.sort(key=lambda x: x[1], reverse=True)
        winner_uri, winner_score, winner_vector = results[0]
        
        if winner_score < CONFIDENCE_THRESHOLD:
            requires_triage = True
            triage_reason = f"Top candidate confidence ({winner_score:.4f}) is below threshold of {CONFIDENCE_THRESHOLD}."

    # 5. ROUTING LOGIC (Auto-Commit vs. Human Triage)
    if not requires_triage:
        print(f"-> [AUTO-COMMIT] Confident match found: {winner_uri} (Score: {winner_score:.4f})")
        print("-" * 75 + "\n")
    else:
        print(f"-> [ROUTED TO TRIAGE] Reason: {triage_reason}")
        print(f"Please select the correct URI context for '{extracted_string}':")
        for idx, uri in enumerate(candidate_uris):
            print(f"  {idx + 1}. {uri}")
            
        while True:
            try:
                choice = int(

--- INITIATING AUTOMATED DISAMBIGUATION (POST-TRAINING) ---

[TRANSACTOR] String: 'Amazon'
Context: "Amazon acquired Whole Foods for $13.7 billion."
-> Selected URI: :AMAZON_COM.organization_entity (Confidence: 1.0000)
    Dimension  Learned_Expectation  Actual_Candidate  Delta (Lower is better)
     physical                  0.1               0.1                      0.0
      bounded                  0.8               0.8                      0.0
     locative                  0.1               0.1                      0.0
      animate                  0.0               0.0                      0.0
     sentient                  0.0               0.0                      0.0
   volitional                  0.8               0.8                      0.0
institutional                  0.9               0.9                      0.0
   collective                  0.8               0.8                      0.0
        telic                  0.4               0.4                      0.0
 